# 🧪 Backtest-quality audit

## 🧭 tl;dr

This notebook audits the research process itself using a moving-average strategy on SPY. It checks:

- one-bar execution timing and a deliberately leaky oracle signal;
- block-bootstrap confidence intervals;
- parameter stability across chronological validation and test periods;
- sensitivity to transaction costs;
- simple ETF-proxy factor exposures;
- a probability-of-backtest-overfitting-style diagnostic.

This is not a guarantee that the strategy is unbiased. It is a compact checklist that shows a beginner where backtests commonly go wrong.

> **Educational research only. Not financial advice.**

## 💡 Why audit a backtest?

A strategy can have a beautiful equity curve and still be invalid. Common causes include:

- using the closing price to trade at that same close;
- choosing parameters after looking at the final test period;
- trying so many variants that one looks good by luck;
- ignoring turnover and slippage;
- mistaking a market beta for a genuine strategy signal.

The shared engine in this repository shifts target positions by one bar. The cells below test that claim directly and then explore the uncertainty that remains.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.backtest import moving_average_signal, run_backtest, split_time_series
from src.data import load_price_data, load_price_panel, make_demo_ohlcv
from src.metrics import calculate_metrics

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.width", 140)

## 📦 Data and baseline

The live sample uses adjusted SPY daily prices from 1993 through 2025. The fallback is deterministic demo data and is labeled as such.

The baseline rule is long when the 50-day moving average is above the 200-day moving average. The signal is created at the close and is held starting on the next observation.

In [2]:
try:
    spy_data = load_price_data("SPY", "1993-01-01", "2026-01-01")
    close = spy_data["close"].dropna().rename("SPY")
    data_source = "Yahoo Finance adjusted SPY daily close"
except Exception as exc:
    close = make_demo_ohlcv(periods=8_200, seed=48, start="1993-01-01")["close"].rename("SPY")
    data_source = f"deterministic synthetic demonstration series ({type(exc).__name__})"

benchmark = close.pct_change().fillna(0.0)
baseline_signal = moving_average_signal(close, fast_window=50, slow_window=200)
baseline = run_backtest(
    close,
    baseline_signal,
    transaction_cost_bps=10.0,
    benchmark_returns=benchmark,
)
print(f"Data source: {data_source}")
print(f"Observations: {len(close):,} | {close.index.min().date()} to {close.index.max().date()}")
display(pd.DataFrame([baseline.metrics], index=["50/200 baseline"])[[
    "annualized_return", "annualized_volatility", "sharpe_ratio",
    "sortino_ratio", "max_drawdown", "annualized_turnover", "number_of_trades",
]])

Data source: Yahoo Finance adjusted SPY daily close
Observations: 8,288 | 1993-01-29 to 2025-12-31


,annualized_return,annualized_volatility,sharpe_ratio,sortino_ratio,max_drawdown,annualized_turnover,number_of_trades
50/200 baseline,0.094382,0.136533,0.729136,1.025893,-0.337173,0.942568,31.0


## ✅ Look-ahead-bias test

The formal check compares the backtester’s held position with target_position.shift(1). If that equality fails, a signal generated at the close could accidentally earn the same close-to-close return.

The second comparison is intentionally educational:

- **Oracle signal:** uses tomorrow’s return to decide today’s position. This is invalid and should not be used.
- **Current-return signal:** uses the already completed return at t, then starts at t+1. This is a valid timing pattern, although it may be economically weak.

A strong result from the oracle is not evidence of a strategy. It is evidence that future information is powerful—and must be excluded.

In [3]:
target = baseline_signal
held = baseline.frame["held_position"]
expected_held = target.shift(1).fillna(0.0)
max_timing_error = float((held - expected_held).abs().max())
assert np.isclose(max_timing_error, 0.0)
print(f"Maximum execution-timing error: {max_timing_error:.3g} ✅")

oracle_signal = close.pct_change().shift(-1).gt(0).astype(float).fillna(0.0)
current_return_signal = close.pct_change().gt(0).astype(float).fillna(0.0)
oracle = run_backtest(close, oracle_signal, transaction_cost_bps=0.0)
current_return = run_backtest(close, current_return_signal, transaction_cost_bps=0.0)

timing_demo = pd.DataFrame([
    {"signal": "50/200 baseline", **baseline.metrics},
    {"signal": "oracle (invalid future information)", **oracle.metrics},
    {"signal": "current return, delayed one bar", **current_return.metrics},
]).set_index("signal")
display(timing_demo[[
    "annualized_return", "annualized_volatility", "sharpe_ratio",
    "max_drawdown", "total_return_after_costs",
]].style.format({"annualized_return": "{:.2%}", "annualized_volatility": "{:.2%}", "sharpe_ratio": "{:.2f}", "max_drawdown": "{:.2%}", "total_return_after_costs": "{:.2%}"}))

Maximum execution-timing error: 0 ✅


,annualized_return,annualized_volatility,sharpe_ratio,max_drawdown,total_return_after_costs
signal,,,,,
50/200 baseline,9.44%,13.65%,0.73,-33.72%,1841.91%
oracle (invalid future information),181.63%,11.53%,9.06,0.00%,61561903040524688.00%
"current return, delayed one bar",2.19%,12.21%,0.24,-67.74%,103.82%


## 🧰 Helpers for chronological comparisons

We evaluate parameter choices on a train/validation/test split. The test period is not used to choose the winning parameter pair.

The metric helper below keeps the benchmark and turnover aligned with the chosen period.

In [4]:
split_frames = split_time_series(pd.DataFrame(index=close.index))

def metrics_for_period(result, period_index):
    period_returns = result.frame["net_returns"].loc[period_index]
    period_benchmark = benchmark.loc[period_index]
    period_positions = result.frame["held_position"].loc[period_index]
    period_turnover = result.frame["turnover"].loc[period_index]
    return dict(calculate_metrics(
        period_returns,
        benchmark_returns=period_benchmark,
        positions=period_positions,
        turnover=period_turnover,
        trade_count=int((period_turnover > 1e-12).sum()),
        annualization=252,
    ))

## 🔬 Parameter stability

We test a small, pre-declared grid of moving-average windows. The validation winner is selected by validation Sharpe, then its test result is read once.

This is not permission to search endlessly. It is a demonstration of how a researcher can separate parameter selection from final evaluation.

In [5]:
candidate_rows = []
for fast_window in [20, 50, 100]:
    for slow_window in [100, 150, 200, 300]:
        if fast_window >= slow_window:
            continue
        signal = moving_average_signal(close, fast_window=fast_window, slow_window=slow_window)
        result = run_backtest(
            close,
            signal,
            transaction_cost_bps=10.0,
            benchmark_returns=benchmark,
        )
        validation = metrics_for_period(result, split_frames["validation"].index)
        test = metrics_for_period(result, split_frames["test"].index)
        candidate_rows.append({
            "fast_window": fast_window,
            "slow_window": slow_window,
            "validation_sharpe": validation["sharpe_ratio"],
            "validation_return": validation["annualized_return"],
            "validation_drawdown": validation["max_drawdown"],
            "test_sharpe": test["sharpe_ratio"],
            "test_return": test["annualized_return"],
            "test_drawdown": test["max_drawdown"],
        })

parameter_grid = pd.DataFrame(candidate_rows)
parameter_grid["validation_rank"] = parameter_grid["validation_sharpe"].rank(ascending=False, method="min")
parameter_grid["test_rank"] = parameter_grid["test_sharpe"].rank(ascending=False, method="min")
selected = parameter_grid.sort_values(
    ["validation_sharpe", "fast_window", "slow_window"],
    ascending=[False, True, True],
).iloc[0]

display(parameter_grid.sort_values("validation_sharpe", ascending=False).style.format({
    "validation_sharpe": "{:.2f}",
    "test_sharpe": "{:.2f}",
    "validation_return": "{:.2%}",
    "validation_drawdown": "{:.2%}",
    "test_return": "{:.2%}",
    "test_drawdown": "{:.2%}",
}))
print(
    f"Selected on validation: fast={int(selected.fast_window)}, slow={int(selected.slow_window)}. "
    f"Its test Sharpe is {selected.test_sharpe:.2f}; the test was not used for selection."
)

,fast_window,slow_window,validation_sharpe,validation_return,validation_drawdown,test_sharpe,test_return,test_drawdown,validation_rank,test_rank
10,100,300,0.96,11.42%,-19.35%,0.87,15.20%,-33.72%,1.000000,1.000000
2,20,200,0.94,10.36%,-14.48%,0.75,10.71%,-34.20%,2.000000,4.000000
5,50,150,0.87,9.68%,-17.35%,0.69,10.26%,-33.72%,3.000000,8.000000
1,20,150,0.84,9.09%,-20.01%,0.78,10.64%,-26.67%,4.000000,3.000000
9,100,200,0.84,9.50%,-19.35%,0.74,12.20%,-33.72%,5.000000,5.000000
3,20,300,0.82,9.32%,-17.15%,0.68,10.44%,-30.92%,6.000000,10.000000
8,100,150,0.82,9.26%,-19.35%,0.69,11.09%,-33.72%,7.000000,9.000000
6,50,200,0.78,8.69%,-19.50%,0.70,10.86%,-33.72%,8.000000,7.000000
4,50,100,0.76,7.95%,-17.96%,0.60,8.32%,-34.36%,9.000000,11.000000
0,20,100,0.70,7.10%,-15.88%,0.82,10.84%,-22.07%,10.000000,2.000000


Selected on validation: fast=100, slow=300. Its test Sharpe is 0.87; the test was not used for selection.


## 🧱 Block-bootstrap confidence intervals

A single Sharpe ratio is an estimate, not a fact. A block bootstrap resamples contiguous 21-trading-day chunks so the resampled series keeps some short-run dependence.

This is a simple uncertainty illustration, not a complete statistical model. It does not correct for all data-snooping, non-stationarity, or parameter-selection bias.

In [6]:
def block_bootstrap_metrics(returns, block_size=21, samples=1000, seed=48):
    values = pd.Series(returns).dropna().to_numpy(dtype=float)
    if len(values) < block_size:
        raise ValueError("Need at least one full bootstrap block")
    rng = np.random.default_rng(seed)
    rows = []
    starts_max = len(values) - block_size
    for _ in range(samples):
        chunks = []
        while sum(len(chunk) for chunk in chunks) < len(values):
            start = int(rng.integers(0, starts_max + 1))
            chunks.append(values[start:start + block_size])
        sample = np.concatenate(chunks)[:len(values)]
        rows.append(calculate_metrics(pd.Series(sample), annualization=252))
    return pd.DataFrame(rows)

chosen_signal = moving_average_signal(
    close,
    fast_window=int(selected.fast_window),
    slow_window=int(selected.slow_window),
)
chosen_result = run_backtest(
    close,
    chosen_signal,
    transaction_cost_bps=10.0,
    benchmark_returns=benchmark,
)
bootstrap = block_bootstrap_metrics(chosen_result.frame["net_returns"].loc[split_frames["test"].index])
confidence = bootstrap[[
    "annualized_return", "sharpe_ratio", "max_drawdown"
]].quantile([0.025, 0.50, 0.975]).rename(index={
    0.025: "2.5%",
    0.50: "median",
    0.975: "97.5%",
})
display(confidence.style.format({"annualized_return": "{:.2%}", "sharpe_ratio": "{:.2f}", "max_drawdown": "{:.2%}"}))
print("Read the interval width as uncertainty around the sample estimate, not as a promise about future returns.")

,annualized_return,sharpe_ratio,max_drawdown
2.5%,1.12%,0.16,-51.85%
median,15.16%,0.90,-28.46%
97.5%,28.32%,1.68,-13.36%


Read the interval width as uncertainty around the sample estimate, not as a promise about future returns.


## 💸 Turnover and slippage sensitivity

The same selected rule is rerun under increasing transaction costs. Fast strategies can lose their appeal because they trade more often; even a slow strategy should show how much return is being spent on implementation friction.

In [7]:
cost_rows = []
for cost in [0, 5, 10, 25, 50]:
    result = run_backtest(
        close,
        chosen_signal,
        transaction_cost_bps=cost,
        benchmark_returns=benchmark,
    )
    cost_rows.append({"cost_bps": cost, **result.metrics})
cost_table = pd.DataFrame(cost_rows).set_index("cost_bps")
display(cost_table[[
    "annualized_return", "annualized_volatility", "sharpe_ratio",
    "max_drawdown", "annualized_turnover", "number_of_trades",
]].style.format({"annualized_return": "{:.2%}", "annualized_volatility": "{:.2%}", "sharpe_ratio": "{:.2f}", "max_drawdown": "{:.2%}", "total_return_after_costs": "{:.2%}"}))

,annualized_return,annualized_volatility,sharpe_ratio,max_drawdown,annualized_turnover,number_of_trades
cost_bps,,,,,,
0,10.89%,14.41%,0.79,-33.72%,0.516892,17.000000
5,10.86%,14.41%,0.79,-33.72%,0.516892,17.000000
10,10.84%,14.41%,0.79,-33.72%,0.516892,17.000000
25,10.75%,14.41%,0.78,-33.72%,0.516892,17.000000
50,10.61%,14.42%,0.77,-33.72%,0.516892,17.000000


## 📊 Simple factor-exposure regression

A trend strategy may still earn much of its result from ordinary market exposure. To make that visible, we regress strategy returns on broad ETF proxies:

- **MKT:** SPY daily return;
- **SMB proxy:** IWM minus SPY;
- **HML proxy:** VTV minus SPY;
- **MOM proxy:** MTUM minus SPY;
- **DEF proxy:** USMV minus SPY.

These are teaching proxies, not canonical academic factor returns. The regression is descriptive and does not prove causality.

In [8]:
FACTOR_TICKERS = ["SPY", "IWM", "VTV", "MTUM", "USMV"]
try:
    factor_panel = load_price_panel(FACTOR_TICKERS, "2006-01-01", "2026-01-01").dropna()
    factor_source = "Yahoo Finance adjusted ETF proxy closes"
except Exception as exc:
    rng = np.random.default_rng(480)
    factor_dates = close.loc["2006-01-01":].index
    factor_panel = pd.DataFrame({
        "SPY": close.loc[factor_dates],
        "IWM": close.loc[factor_dates] * np.exp(np.cumsum(rng.normal(0, 0.001, len(factor_dates)))),
        "VTV": close.loc[factor_dates] * np.exp(np.cumsum(rng.normal(0, 0.0008, len(factor_dates)))),
        "MTUM": close.loc[factor_dates] * np.exp(np.cumsum(rng.normal(0, 0.0012, len(factor_dates)))),
        "USMV": close.loc[factor_dates] * np.exp(np.cumsum(rng.normal(0, 0.0007, len(factor_dates)))),
    })
    factor_source = f"deterministic synthetic factor demonstration ({type(exc).__name__})"

asset_returns = factor_panel.pct_change()
market = asset_returns["SPY"].rename("MKT")
factor_returns = pd.DataFrame({
    "MKT": market,
    "SMB_proxy": asset_returns["IWM"] - market,
    "HML_proxy": asset_returns["VTV"] - market,
    "MOM_proxy": asset_returns["MTUM"] - market,
    "DEF_proxy": asset_returns["USMV"] - market,
})
regression_data = pd.concat([
    chosen_result.frame["net_returns"].rename("strategy"),
    factor_returns,
], axis=1).dropna()

X = np.column_stack([np.ones(len(regression_data)), regression_data[factor_returns.columns].to_numpy()])
y = regression_data["strategy"].to_numpy()
coefficients = np.linalg.lstsq(X, y, rcond=None)[0]
fitted = X @ coefficients
residual = y - fitted
r_squared = 1 - (residual @ residual) / ((y - y.mean()) @ (y - y.mean()))
exposure_table = pd.DataFrame({
    "coefficient": coefficients,
    "label": ["alpha"] + list(factor_returns.columns),
}, index=["alpha"] + list(factor_returns.columns))
exposure_table.loc["R-squared"] = {"coefficient": r_squared, "label": "share of daily variance explained"}
print(f"Factor source: {factor_source}")
display(exposure_table.style.format({"coefficient": "{:.4f}"}))

Factor source: Yahoo Finance adjusted ETF proxy closes


,coefficient,label
alpha,0.0000,alpha
MKT,0.8276,MKT
SMB_proxy,-0.0406,SMB_proxy
HML_proxy,0.2129,HML_proxy
MOM_proxy,0.2466,MOM_proxy
DEF_proxy,-0.0815,DEF_proxy
R-squared,0.8418,share of daily variance explained


## 🎲 Probability-of-backtest-overfitting-style diagnostic

A formal Probability of Backtest Overfitting analysis uses many combinations of in-sample and out-of-sample paths plus explicit assumptions about the selection process. This notebook provides a clearly labeled classroom proxy:

1. rank the parameter candidates by validation Sharpe;
2. rank the same candidates by test Sharpe;
3. measure how often candidates in the better validation half fall below the median test result;
4. report the rank correlation between validation and test.

If validation and test rankings disagree, parameter selection is unstable. The number below is **not** a formal PBO estimate.

In [9]:
pbo_table = parameter_grid.copy()
pbo_table["validation_percentile"] = pbo_table["validation_sharpe"].rank(pct=True)
pbo_table["test_percentile"] = pbo_table["test_sharpe"].rank(pct=True)
validation_top_half = pbo_table["validation_percentile"] >= 0.50
pbo_proxy = float((pbo_table.loc[validation_top_half, "test_percentile"] < 0.50).mean())
validation_ranks = pbo_table["validation_sharpe"].rank()
test_ranks = pbo_table["test_sharpe"].rank()
rank_correlation = float(validation_ranks.corr(test_ranks))
pbo_summary = pd.DataFrame([{
    "candidate_parameter_sets": len(pbo_table),
    "validation_test_rank_correlation": rank_correlation,
    "PBO_like_proxy": pbo_proxy,
    "validation_winner_test_sharpe": float(selected.test_sharpe),
    "median_candidate_test_sharpe": float(pbo_table["test_sharpe"].median()),
}])
display(pbo_summary.style.format({
    "validation_test_rank_correlation": "{:.2f}",
    "PBO_like_proxy": "{:.2%}",
    "validation_winner_test_sharpe": "{:.2f}",
    "median_candidate_test_sharpe": "{:.2f}",
}))
print("Interpretation: a high proxy means validation winners often land below the test median. Treat it as a warning light, not a probability theorem.")

,candidate_parameter_sets,validation_test_rank_correlation,PBO_like_proxy,validation_winner_test_sharpe,median_candidate_test_sharpe
0,11,0.35,33.33%,0.87,0.73


Interpretation: a high proxy means validation winners often land below the test median. Treat it as a warning light, not a probability theorem.


## 🧠 What went well and what went wrong?

**What went well**

- The execution-timing assertion verifies the core no-look-ahead contract.
- The oracle comparison makes the danger of future information concrete.
- Parameter selection is separated from the untouched test period.
- Bootstrap intervals, cost curves, and factor exposures make the headline Sharpe harder to overstate.
- The overfitting diagnostic reports instability rather than hiding it.

**What went wrong or remains incomplete**

- A 50/200 grid is still a small and researcher-chosen search space.
- Block bootstrap confidence intervals are only approximate.
- ETF factor proxies are not the same as stock-level Fama–French factors.
- The PBO-like statistic is educational, not formal Bailey–Lo–MacLean PBO.
- A stronger audit would use multiple markets, walk-forward re-estimation, purged/embargoed cross-validation where appropriate, a deflated Sharpe ratio, and point-in-time data.

The quality lesson is simple: a backtest is a chain of assumptions. Audit the chain, not just the final equity curve.